In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.preprocess import (
    load_data,
    clean_data,
    handle_missing_values
)

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [2]:
df = load_data("../data/customer_churn_1M.csv")

df = clean_data(df)

df = handle_missing_values(df)

print(df.shape)

(1000000, 31)


In [3]:
top_features = [

    "customer_satisfaction",
    "num_service_calls",
    "num_complaints",
    "monthlycharges",
    "totalcharges",
    "days_since_signup",
    "annual_income",
    "credit_score",
    "contract",
    "days_since_last_interaction"

]

In [4]:
X = df[top_features]

y = df["churn"]

print(X.shape)

(1000000, 10)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
categorical_cols = ["contract"]

numerical_cols = [

    "customer_satisfaction",
    "num_service_calls",
    "num_complaints",
    "monthlycharges",
    "totalcharges",
    "days_since_signup",
    "annual_income",
    "credit_score",
    "days_since_last_interaction"

]

In [7]:
categorical_cols = ["contract"]

numerical_cols = [

    "customer_satisfaction",
    "num_service_calls",
    "num_complaints",
    "monthlycharges",
    "totalcharges",
    "days_since_signup",
    "annual_income",
    "credit_score",
    "days_since_last_interaction"

]

In [9]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [10]:
rf_top10 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=15,
                min_samples_split=10,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [11]:
rf_top10.fit(X_train, y_train)

print("Training Complete")

Training Complete


In [12]:
y_pred = rf_top10.predict(X_test)

y_prob = rf_top10.predict_proba(X_test)[:, 1]

In [16]:
print("Classification Report:")
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.65      0.77    180155
           1       0.16      0.57      0.24     19845

    accuracy                           0.65    200000
   macro avg       0.54      0.61      0.51    200000
weighted avg       0.86      0.65      0.72    200000

Confusion Matrix:
[[117997  62158]
 [  8440  11405]]
ROC-AUC: 0.6640328922550449


## Top Features


contract_two_year

customer_satisfaction

contract_one_year

num_service_calls

num_complaints

monthlycharges

totalcharges

days_since_signup

annual_income

credit_score

In [ ]:
import joblib
import os

# Ensure the models directory exists
os.makedirs("../models", exist_ok=True)

# Save the model
joblib.dump(
    rf_top10,
    "../models/rf_top10_features.pkl"
)

print("Top 10 RF model saved")

NameError: name 'rf_top10' is not defined